<a href="https://colab.research.google.com/github/nawrotlab/Nawrot_CNS_Course_solutions/blob/main/Neural_Data_Analysis_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Basic analyses of intracellular recordings: From spontaneous to evoked events. - Solutions

## Synopsis
This course module introduces basic analyses of intracellular recordings from rat
neocortical cells in vivo and in vitro. Part I analyzes action potentials from sharp
intracellular voltage recordings under anaesthetized conditions. Parts II and III
introduce the analysis of data from intracellular patch-clamp recordings in the voltage
clamp mode, focusing on the detection and statistical description of spontaneous as
well as evoked post-synaptic currents (PSCs). After having completed the module,
students should be able to detect events in intracellular recordings by setting
appropriate thresholds and analyze event-triggered amplitude distribution, reliability
and amplitude variability. Data from one in vivo intracellular recording from rat
somatosensory cortex is provided. Data from whole-cell patch-clamp recordings from
layer-V pyramidal cells of rat somatosensory cortex are provided, where
spontaneously occurring PSCs are used to train detection and assessment of
amplitude distribution, while PSCs evoked via glutamate uncaging at presynaptic
sites are used for reliability and amplitude variability measures.

## Introduction
Intracellular recordings *in vivo* are difficult to achieve and typically performed in head-fixed or
anaesthetized animals. They allow to analyze subthreshold membrane potential fluctuations, spike initiation, and spike wave form. Over the past decades electrophysiological experiments in vitro have contributed strongly to
our knowledge of neural function on the level of single neurons (e.g. dendritic integration and
computation), synaptic transmission (e.g. synaptic strength, variability and connectivity) and
synaptic plasticity (e.g. LTP, STP, STDP). Here we use intracellular recordings with sharp pipettes from the
somatosensory cortex of the anaesthetized rat [1,2].   
There are different types of *in vitro* preparations, divided into two broad classes: cultured nerve tissue and acute brain slices. Compared to experiments in vivo these preparations have the major advantage of good experimental
accessibility with a variety of different electrophysiological techniques. This exercise covers the analysis of intracellular whole-cell patch-clamp recordings from layer V pyramidal cells in an acute slice of rat somatosensory cortex. The patch-clamp electrode is a glass pipette filled with an electrolyte, a solution that resembles the intracellular medium in its ionic composition, to provide for electric coupling of the amplifier equipment to
the cytoplasm of the cell. With this electrode, it is possible to measure the membrane potential relative to the reference electrode, which is connected to ground. Therefore, the extracellular potential outside the neurons is defined as zero. The recordings for today’s exercise were made in the voltage-clamp mode, which means that the trans-membrane
voltage is kept constant by means of injecting currents that counteract all synaptic currents. By measuring the dynamic current that is injected we effectively measured the (dynamic) synaptic current that flows into the cell in reaction to vesicle release at synaptic sites, which may have occurred spontaneously or in reaction to a presynaptic action potential.  
A secondary goal of the exercises is to explore visualization techniques in Python (matplotlib) and to produce ‘camera-ready’ figures that combine different graphs in one figure. Make sure that all axes have proper labels (dimension and units).

General advice: all datasets for this notebook live in `data/neural_data_analysis/`. In Colab, run the setup cell below once; it clones the repository and defines `DATA_DIR`, `A1_PATH`, `SPONTANEOUS_PATH`, and `RELIABILITY_PATH`.

The `_nostruct.mat` files are the easiest entry point in Python because `scipy.io.loadmat` returns the relevant arrays directly as dictionary entries. For example:

```python
from scipy.io import loadmat
mat = loadmat(A1_PATH)
V = mat["V"].ravel()
dt = float(mat["SampleIntervalSeconds"].squeeze())
```

For the other two datasets, the most relevant keys are `I`, `FI`, `Z`, `TimeResolutionS`, `I_TimeResolutionS`, and `Z_TimeResolutionS`.


In [ ]:

from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/schmitfe/Nawrot_CNS_Course.git"
REPO_NAME = "Nawrot_CNS_Course"


def prepare_repo() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "data").is_dir():
        return cwd

    clone_parent = Path("/content") if "google.colab" in sys.modules else cwd
    repo_root = clone_parent / REPO_NAME
    if not repo_root.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_root)], check=True)
    os.chdir(repo_root)
    return repo_root.resolve()


REPO_ROOT = prepare_repo()
DATA_DIR = REPO_ROOT / "data" / "neural_data_analysis"
A1_PATH = DATA_DIR / "A1_data_set_040528_boucsein_nostruct.mat"
SPONTANEOUS_PATH = DATA_DIR / "spontaneous_recording_nostruct.mat"
RELIABILITY_PATH = DATA_DIR / "synaptic_response_variability_nostruct.mat"
print(f"REPO_ROOT = {REPO_ROOT}")
print(f"DATA_DIR = {DATA_DIR}")


In [ ]:
from scipy.io import loadmat
import matplotlib.pyplot as plt
import numpy as np


def event_cutouts(signal, event_idx, before, after):
    valid = event_idx[(event_idx >= before) & (event_idx + after < len(signal))]
    return np.array([signal[i - before:i + after] for i in valid])


## Part I : Intracellular membrane potential and action potential waveform
The neocortical network is permanently active. Goal of this first part is to detect spontaneous
intracellular activity in the anesthetized neocortex. Can we see the postsynaptic potentials due
to synaptic input? What is the average membrane potential in between action potentials? Is
there a fixed voltage threshold for spike initiation? How variable is the action potential
waveform?

Create a new code cell below each task and write your solution there. When you run the notebook from top to bottom, it should reproduce your Figures 1-4.


1.1 Load `A1_PATH`, which points to `A1_data_set_040528_boucsein_nostruct.mat`. This file contains an in vivo intracellular recording from a cortical neuron in millivolts (mV). Create a new figure with `plt.figure()` and plot only a short piece (e.g. 2 s) of the complete voltage trace. Set the y-axis range to `(-75, 20)` mV. Compare your plot to Fig. 1A in [1]. How long is the complete voltage trace in seconds?


1.2 Determine all time points $t_i$ , $1≤i≤N$ at which an action potential occurred. As a first step you need to define a suitable threshold $\theta$. Then detect the spike times as positive threshold crossings (i.e. the first data point of a spike that is higher than the threshold, check out `np.diff`). Assign a variable to the resulting list of spike times. How many action potentials did you detect? What is the average firing rate (i.e. action potentials per second) of this neuron?

1.3 Now determine the average waveform of an action potential. Loop across all AP onset times `t_i`, extract a cutout of about ±10 ms around each onset, and average across all cutouts. Create a new figure with `plt.figure()` and plot the resulting average spike waveform. Optional: add the average waveform ± 1 standard deviation across all APs.


1.4 What are the mean and variance of the membrane potential distribution when excluding the action-potential cutouts? Create a new Figure 3 and plot a histogram of the membrane-potential distribution. In Python you can use `np.histogram` or `plt.hist`; define the bin edges explicitly so the histogram resolution is under your control.


1.5 Optional: The standard integrate-and-fire (I&F) model assumes a fixed voltage threshold for spike initiation. Can you determine a fixed voltage threshold that predicts the onset of each spike? Repeat Exercise 1.3 with a longer cutout of 500 ms before spike detection. Create a new Figure 4. Plot the mean action potential in red and, in addition, plot mean ± standard deviation.


### Solution: Part I

In [ ]:
a1 = loadmat(A1_PATH, squeeze_me=True)
V = a1["V"].ravel()
dt = float(a1["SampleIntervalSeconds"])
t = np.arange(V.size) * dt
print(f"complete duration: {t[-1]:.2f} s")

plt.figure()
plt.plot(t[t < 2], V[t < 2])
plt.ylim(-75, 20)
plt.xlabel("time (s)")
plt.ylabel("membrane potential (mV)")
plt.title("Figure 1: intracellular voltage")
plt.show()

threshold = -40
spike_idx = np.where((V[1:] >= threshold) & (V[:-1] < threshold))[0] + 1
spike_times = spike_idx * dt
print("spikes:", len(spike_idx), "mean firing rate (Hz):", len(spike_idx) / t[-1])

before = after = round(0.010 / dt)
spikes = event_cutouts(V, spike_idx, before, after)
spike_t = (np.arange(spikes.shape[1]) - before) * dt * 1000
spike_mean = spikes.mean(axis=0)
spike_std = spikes.std(axis=0)
plt.figure()
plt.plot(spike_t, spike_mean, color="red")
plt.fill_between(spike_t, spike_mean - spike_std, spike_mean + spike_std, alpha=0.3)
plt.xlabel("time from threshold crossing (ms)")
plt.ylabel("membrane potential (mV)")
plt.title("Figure 2: average action potential")
plt.show()

non_spike_mask = np.ones(V.size, dtype=bool)
for i in spike_idx:
    non_spike_mask[max(0, i - before):min(V.size, i + after)] = False
non_spike_V = V[non_spike_mask]
print("non-spiking mean:", non_spike_V.mean(), "variance:", non_spike_V.var())
plt.figure()
plt.hist(non_spike_V, bins=np.arange(-75, -20, 0.5))
plt.xlabel("membrane potential (mV)")
plt.ylabel("samples")
plt.title("Figure 3: membrane-potential distribution")
plt.show()

long_before = round(0.5 / dt)
long_spikes = event_cutouts(V, spike_idx, long_before, after)
long_t = (np.arange(long_spikes.shape[1]) - long_before) * dt * 1000
plt.figure()
plt.plot(long_t, long_spikes.mean(axis=0), color="red")
plt.fill_between(
    long_t,
    long_spikes.mean(axis=0) - long_spikes.std(axis=0),
    long_spikes.mean(axis=0) + long_spikes.std(axis=0),
    alpha=0.3,
)
plt.xlabel("time from threshold crossing (ms)")
plt.ylabel("membrane potential (mV)")
plt.title("Figure 4: variability before spike onset")
plt.show()


1.6 Do all your graphs have proper axis labels and units? Export your result figures with `plt.savefig(...)` in a suitable format.


## Part II : Spontaneous synaptic events
In neocortical acute brain slices, there is only little spontaneous activity compared to the in
vivo situation due to the de-afferentiation (cutting off the inputs from the rest of the brain).
This is an advantage if we want to determine the size of single excitatory postsynaptic
currents (EPSCs). Goal of the second part of the exercise is to first determine the average
time course and peak amplitude of synaptic events that arrive at one particular cell, and,
secondly, to determine the distribution of peak amplitudes that stem from different synaptic
release sites.

2.1 Load `SPONTANEOUS_PATH`, which points to `spontaneous_recording_nostruct.mat`.

After `mat = loadmat(SPONTANEOUS_PATH)`, the most important entries are:
- `I`: unfiltered current trace
- `FI`: band-pass filtered current trace
- `TimeResolutionS`: temporal resolution in seconds
- `SamplingFrequencyHz`: sampling frequency in Hz


2.2 Create a figure and plot the raw signal (`I`) and the band-pass filtered signal (`FI`) in two different colors on top of each other (graph 1). Use an appropriate time scale (s or ms) on the x-axis and choose a time window that reveals the structure of the data (e.g. 1 s). The arrays are long, so first inspect their `.shape` or length before plotting the full trace.


2.3 Use a threshold to detect PSCs in the band-pass filtered data. The threshold should be chosen such that only obvious synaptic events cross the threshold, while noise fluctuations do not reach this threshold. Add the horizontal threshold line to graph 1. Because PSCs are downward deflections in this recording, look for the **first sample below threshold** for which the previous sample was still above threshold. Assign the variable `onset_idx` to the vector indices at which these threshold crossings occur. How many PSCs did you detect? How many did your neighbors detect? What is the reason for differing numbers?

![Threshold crossing detection](assets/threshold_crossing_detection.png)


2.4 Make data cutouts of stretches of the raw signal around the onsets of the detected PSCs. Choose a reasonable time interval before and after the onset such that the time course of the PSCs is well captured. Assign the variable “PSCs” to the matrix of cutouts.

2.5 Graph 2: plot all PSCs in one axis. On top plot the average PSC with a stronger line. Use a second axis (graph 3) to plot the average PSC and two additional lines indicating the standard deviation in positive and negative direction with respect to the average PSC. What is the peak amplitude of the avg. PSC?

2.6 Statistics: Determine the peak amplitude of each single PSC and produce a histogram (i.e. show the distribution) of PSC peak amplitudes (graph 3). Is this distribution symmetric or skewed? Produce a histogram on the logarithmic axes, or, equivalently, produce a histogram of the logarithmic values of peak amplitude. Distribution of synaptic strength is well approximated by the so-called log-normal distribution (i.e. normally distributed on the logarithmic axis).

### Solution: Part II

In [ ]:
spontaneous = loadmat(SPONTANEOUS_PATH, squeeze_me=True)
I = spontaneous["I"].ravel()
FI = spontaneous["FI"].ravel()
dt_I = float(spontaneous["TimeResolutionS"])
t_I = np.arange(I.size) * dt_I

threshold_psc = -8
onset_idx = np.where((FI[1:] < threshold_psc) & (FI[:-1] >= threshold_psc))[0] + 1
print("detected PSCs:", len(onset_idx))

window = t_I < 1
plt.figure()
plt.plot(t_I[window], I[window], label="raw")
plt.plot(t_I[window], FI[window], label="filtered")
plt.axhline(threshold_psc, color="red", label="threshold")
plt.xlabel("time (s)")
plt.ylabel("current (pA)")
plt.legend()
plt.show()

psc_before = round(0.020 / dt_I)
psc_after = round(0.080 / dt_I)
PSCs = event_cutouts(I, onset_idx, psc_before, psc_after)
psc_t = (np.arange(PSCs.shape[1]) - psc_before) * dt_I * 1000
mean_psc = PSCs.mean(axis=0)
std_psc = PSCs.std(axis=0)
peak_amplitudes = -PSCs.min(axis=1)
print("average PSC peak amplitude (pA):", -mean_psc.min())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(psc_t, PSCs.T, alpha=0.15)
axes[0].plot(psc_t, mean_psc, color="black", linewidth=2)
axes[1].plot(psc_t, mean_psc, color="black")
axes[1].fill_between(psc_t, mean_psc - std_psc, mean_psc + std_psc, alpha=0.3)
axes[2].hist(peak_amplitudes[peak_amplitudes > 0], bins=25)
axes[2].set_xscale("log")
for ax in axes[:2]:
    ax.set_xlabel("time from onset (ms)")
    ax.set_ylabel("current (pA)")
axes[2].set_xlabel("PSC peak amplitude (pA, log scale)")
axes[2].set_ylabel("count")
plt.tight_layout()
plt.show()


2.7 Do all your graphs have proper axes labeling and units? You should save your result figures using the `plt.savefig` function.

## Part III : Reliability of synaptic transmission
In this part we analyze a different experimental paradigm. Again, voltage clamp
measurements from a layer V pyramidal neuron were performed to observe membrane
currents. To measure the repeated postsynaptic response to synaptic transmission, a
presynaptic neuron was repeatedly stimulated by means of phototlytic release of the
excitatory neurotransmitter glutamate using short pulses (4-10ms) of UV light [3]. For
generating the pulses a shutter (similar to that in a photo camera) was used to start and end
illumination of small spots of the slice by a continuous UV laser beam. The movement of the
mechanical shutter was recorded along with the physiological data. This allows to determine
the exact times at which the shutter was opened. These shutter opening times will serve us
as the trigger (instead of the threshold crossing in Part II) for cutting out the data pieces. The
goal is to determine the trial-to-trial variation of the peak amplitude of the PSCs and the
temporal precision of PSC onset.

3.1 Load `RELIABILITY_PATH`, which points to `synaptic_response_variability_nostruct.mat`.

After `mat = loadmat(RELIABILITY_PATH)`, the most important entries are:
- `I`: current trace
- `Z`: shutter command voltage
- `I_TimeResolutionS`: temporal resolution of the current trace in seconds
- `Z_TimeResolutionS`: temporal resolution of the shutter command voltage in seconds


3.2 Detect the stimulation onset indices/times from the variable `Z`. The shutter is closed when the shutter control voltage is `+5 V` and open when it assumes `-2 V`. Note that the shutter command voltage was sampled with a different time resolution than the current trace. There are 42 shutter openings corresponding to 3 different presynaptic stimulation sites (3 different presynaptic neurons that were targeted) and 14 repetitions for each target site. The stimulation sequence was `target 1, target 2, target 3, target 1, ...` and so on. A robust strategy is: first detect **all 42 opening times in order**, then reshape that one-dimensional list into repeated triplets and finally regroup them into one array per target site. For example, a reshape to `(14, 3)` followed by a transpose gives a convenient `(3, 14)` organization.


3.3 Amplitude variability: For each presynaptic site, extract the responses to the stimulation during 100 ms following the stimulation (cutouts) and make for each target a graph (graph 1-3) that shows all 14 repetitions in different colors. Measure the peak amplitude for each single repetition and calculate mean and standard deviation of
peak amplitudes across repetitions, as well as the coefficient of variation CV (std / mean). Add the numbers as text to your graph. How does the CV depend on the avg. peak amplitude for the three cases? Do your results fit to the results given in Figure 5C in [3]?

3.4 Temporal precision. Assess the temporal precision with which the PSCs occurred in repeated trials. An easy way to do this is to detect the PSC onset again by using a threshold and then measure the standard deviation of onset times (in milliseconds). The smaller this standard deviation, the more precise is the response.

### Solution: Part III

In [ ]:
reliability = loadmat(RELIABILITY_PATH, squeeze_me=True)
current = reliability["I"].ravel()
shutter = reliability["Z"].ravel()
dt_current = float(reliability["I_TimeResolutionS"])
dt_shutter = float(reliability["Z_TimeResolutionS"])

opening_idx = np.where((shutter[1:] < 0) & (shutter[:-1] >= 0))[0] + 1
opening_times = opening_idx * dt_shutter
opening_times = opening_times.reshape(14, 3).T
print("shutter openings:", opening_times.size)

n_response = round(0.100 / dt_current)
responses = np.empty((3, 14, n_response))
for site in range(3):
    for trial in range(14):
        start = round(opening_times[site, trial] / dt_current)
        responses[site, trial] = current[start:start + n_response]

response_t = np.arange(n_response) * dt_current * 1000
peak = responses.max(axis=2)
peak_mean = peak.mean(axis=1)
peak_std = peak.std(axis=1)
cv = peak_std / peak_mean

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True, sharey=True)
for site, ax in enumerate(axes):
    ax.plot(response_t, responses[site].T, alpha=0.7)
    ax.set_title(f"site {site + 1}: mean={peak_mean[site]:.1f}, CV={cv[site]:.2f}")
    ax.set_xlabel("time after stimulation (ms)")
axes[0].set_ylabel("current (pA)")
plt.tight_layout()
plt.show()

onset_threshold = 40
onset_times = np.full((3, 14), np.nan)
for site in range(3):
    for trial in range(14):
        trace = responses[site, trial]
        crossings = np.where((trace[1:] >= onset_threshold) & (trace[:-1] < onset_threshold))[0] + 1
        if crossings.size:
            onset_times[site, trial] = crossings[0] * dt_current * 1000
print("onset-time standard deviations (ms):", np.nanstd(onset_times, axis=1))


***

## Data Sets

A1 – intracellular recording of spontaneous activity in a cortical cell of the anaesthetized rat. Details are given in Nawrot et al. (2007). These experiments were performed by Clemens Boucsein, University of Freiburg, Germany and Dymphie Suchanek, University of Freiburg, Germany.

    Recommended Python file: A1_data_set_040528_boucsein_nostruct.mat
B – data from intracellular voltage-clamp recordings (unpublished) from one layer 5 pyramidal cell in the acute neocortical slice from rat somatosensory cortex. Data recordings by C. Boucsein and M. Nawrot.

    Recommended Python file: spontaneous_recording_nostruct.mat
C – data from laser uncaging experiments as published in Boucsein et al. (2005) and Boucsein et al. (2011). This data set contains voltage-clamp recordings from one layer 5 pyramidal cell in the acute neocortical slice from rat somatosensory cortex. Presynaptic neurons at three different target sites were stimulated by focused photolysis of caged glutamate. These neurons produced action potentials, which resulted in excitatory postsynaptic currents (EPSCs) in the postsynaptic cell.

    Recommended Python file: synaptic_response_variability_nostruct.mat


## References:
[1] Nawrot MP, Boucsein C, Rodriguez-Molina V, Aertsen A, Grün S, Rotter S (2007) Serial
interval statistics of spontaneous activity in cortical neurons in vivo and in vitro.
Neurocomputing 70: 1717-1722

[2] Fucke T, Suchanek D, Nawrot MP, Seamari Y, Heck DH, Aertsen A, Boucsein C (2011).
Stereotypical spatiotemporal activity patterns during slow-wave activity in the
neocortex. Journal of neurophysiology, 106(6), 3035-3044.

[3] Boucsein C, Nawrot M, Rotter S, Aertsen A, Heck D (2005) Controlling Synaptic input
Patterns In vitro by Dynamic Photo Stimulation. J Neurophysiol 94: 2948-2958

[4] Markram H, Lubke J, Frotscher M, Roth A, Sakmann B (1997) Physiology and anatomy of
synaptic connections between thick tufted pyramidal neurons in the developing rat neocortex.
J Physiol 500 (2):409-440.

[5] Boucsein C, Nawrot M, Schnepel P, Aertsen A (2011). Beyond the cortical column:
abundance and physiology of horizontal connections imply a strong role for inputs from the
surround. Frontiers in Neuroscience, 5, 32.

[6] Nawrot MP, Schnepel P, Aertsen A, Boucsein C (2009) Precisely timed signal
transmission in neocortical networks with reliable intermediate-range projections. Frontiers in
Neural Circuits 3:1 http://frontiersin.org/neuralcircuits/paper/10.3389/neuro.04/001.2009/pdf/